In [1]:
#importing libraried
import pandas as pd
from sklearn.preprocessing import MinMaxScaler , LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score ,confusion_matrix,precision_score,recall_score,f1_score,classification_report
from sklearn.model_selection import train_test_split,GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#loading dataset
df=pd.read_csv("Global_Pollution_Analysis - Global_Pollution_Analysis.csv")

In [3]:
df.head()

,Country,Year,Air_Pollution_Index,Water_Pollution_Index,Soil_Pollution_Index,Industrial_Waste (in tons),Energy_Recovered (in GWh),CO2_Emissions (in MT),Renewable_Energy (%),Plastic_Waste_Produced (in tons),Energy_Consumption_Per_Capita (in MWh),Population (in millions),GDP_Per_Capita (in USD)
0,Hungary,2005,272.70,124.27,51.95,94802.83,158.14,5.30,41.11,37078.88,12.56,42.22,20972.96
1,Singapore,2001,86.72,60.34,117.22,56283.92,498.04,6.34,36.44,33128.20,5.23,137.25,34850.41
2,Romania,2016,91.59,83.36,121.72,56256.02,489.51,49.69,9.38,18803.46,13.15,124.47,57773.15
3,Cook Islands,2018,280.61,67.16,93.58,74864.73,145.18,8.91,18.97,9182.27,0.78,67.80,21837.51
4,Djibouti,2008,179.16,127.53,121.55,76862.06,40.38,14.93,34.00,39235.12,12.84,186.52,41379.37


In [4]:
df.isnull().sum()

Country                                   0
Year                                      0
Air_Pollution_Index                       0
Water_Pollution_Index                     0
Soil_Pollution_Index                      0
Industrial_Waste (in tons)                0
Energy_Recovered (in GWh)                 0
CO2_Emissions (in MT)                     0
Renewable_Energy (%)                      0
Plastic_Waste_Produced (in tons)          0
Energy_Consumption_Per_Capita (in MWh)    0
Population (in millions)                  0
GDP_Per_Capita (in USD)                   0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Country                                 200 non-null    object 
 1   Year                                    200 non-null    int64  
 2   Air_Pollution_Index                     200 non-null    float64
 3   Water_Pollution_Index                   200 non-null    float64
 4   Soil_Pollution_Index                    200 non-null    float64
 5   Industrial_Waste (in tons)              200 non-null    float64
 6   Energy_Recovered (in GWh)               200 non-null    float64
 7   CO2_Emissions (in MT)                   200 non-null    float64
 8   Renewable_Energy (%)                    200 non-null    float64
 9   Plastic_Waste_Produced (in tons)        200 non-null    float64
 10  Energy_Consumption_Per_Capita (in MWh)  200 non-null    float6

In [6]:
df=df.drop(columns=["Country"])

In [7]:
df.head()

,Year,Air_Pollution_Index,Water_Pollution_Index,Soil_Pollution_Index,Industrial_Waste (in tons),Energy_Recovered (in GWh),CO2_Emissions (in MT),Renewable_Energy (%),Plastic_Waste_Produced (in tons),Energy_Consumption_Per_Capita (in MWh),Population (in millions),GDP_Per_Capita (in USD)
0,2005,272.70,124.27,51.95,94802.83,158.14,5.30,41.11,37078.88,12.56,42.22,20972.96
1,2001,86.72,60.34,117.22,56283.92,498.04,6.34,36.44,33128.20,5.23,137.25,34850.41
2,2016,91.59,83.36,121.72,56256.02,489.51,49.69,9.38,18803.46,13.15,124.47,57773.15
3,2018,280.61,67.16,93.58,74864.73,145.18,8.91,18.97,9182.27,0.78,67.80,21837.51
4,2008,179.16,127.53,121.55,76862.06,40.38,14.93,34.00,39235.12,12.84,186.52,41379.37


In [8]:
df["Air_Pollution_Index"].describe()

count    200.00000
mean     180.62695
std       67.07331
min       50.30000
25%      134.97250
50%      183.38500
75%      237.42500
max      297.95000
Name: Air_Pollution_Index, dtype: float64

In [9]:
df["Water_Pollution_Index"].describe()

count    200.000000
mean     115.068100
std       47.580911
min       31.130000
25%       74.550000
50%      112.305000
75%      157.477500
max      199.320000
Name: Water_Pollution_Index, dtype: float64

In [10]:
df["Soil_Pollution_Index"].describe()

count    200.000000
mean      76.488550
std       39.692727
min       11.150000
25%       40.895000
50%       78.600000
75%      109.212500
max      149.230000
Name: Soil_Pollution_Index, dtype: float64

In [11]:
low_th = df[['Air_Pollution_Index',
             'Water_Pollution_Index',
             'Soil_Pollution_Index']].mean(axis=1).quantile(0.33)

high_th = df[['Air_Pollution_Index',
              'Water_Pollution_Index',
              'Soil_Pollution_Index']].mean(axis=1).quantile(0.66)

In [12]:
def classify_pollution(row):
    score = (row['Air_Pollution_Index'] + 
             row['Water_Pollution_Index'] + 
             row['Soil_Pollution_Index']) / 3
    
    if score <= low_th:
        return 0   
    elif score <= high_th:
        return 1   
    else:
        return 2  

df['pollution_severity'] = df.apply(classify_pollution, axis=1)

In [13]:
print(df['pollution_severity'].value_counts())

pollution_severity
2    68
0    66
1    66
Name: count, dtype: int64


In [14]:
x=df.drop(columns=["pollution_severity","Year"])
y=df["pollution_severity"]

In [15]:
scaler = MinMaxScaler()
x=scaler.fit_transform(x)



In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.2,random_state=42)

In [17]:
model = MultinomialNB()
model.fit(x_train, y_train)

MultinomialNB()

In [18]:
pred_= model.predict(x_test)

In [19]:
print("Accuracy :",accuracy_score(y_test, pred_))

Accuracy : 0.35


In [20]:
print("Precision:", precision_score(y_test, pred_,average='macro'))

Precision: 0.29258098223615464


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
print(y.value_counts())

pollution_severity
2    68
0    66
1    66
Name: count, dtype: int64


In [22]:
print("Recall   :", recall_score(y_test, pred_,average='macro'))

Recall   : 0.48611111111111116


In [23]:
print("F1-score :", f1_score(y_test, pred_,average='macro'))

F1-score : 0.3290246768507638


In [24]:
print("confusion matrix :\n", confusion_matrix(y_test, pred_))

confusion matrix :
 [[ 7  0  5]
 [ 3  0 17]
 [ 1  0  7]]


In [25]:
print(classification_report(y_test, pred_))

              precision    recall  f1-score   support

           0       0.64      0.58      0.61        12
           1       0.00      0.00      0.00        20
           2       0.24      0.88      0.38         8

    accuracy                           0.35        40
   macro avg       0.29      0.49      0.33        40
weighted avg       0.24      0.35      0.26        40



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
from sklearn.neighbors import KNeighborsClassifier
param_grid = {'n_neighbors': range(1, 31)}
grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,               
    scoring='accuracy')
grid.fit(x_train, y_train)
print("Best K:", grid.best_params_)
best_knn = grid.best_estimator_
pred=best_knn.predict(x_test)

Best K: {'n_neighbors': 15}


In [27]:
print("Accuracy :",accuracy_score(y_test, pred))

Accuracy : 0.5


In [29]:
print("Precision:", precision_score(y_test, pred,average='macro'))

Precision: 0.5213507625272332


In [30]:
print("Recall   :", recall_score(y_test, pred,average='macro'))

Recall   : 0.6055555555555555


In [31]:
print("F1-score :", f1_score(y_test, pred,average='macro'))

F1-score : 0.4844444444444444


In [32]:
print("confusion matrix :\n", confusion_matrix(y_test, pred))

confusion matrix :
 [[11  0  1]
 [ 7  3 10]
 [ 0  2  6]]


In [33]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.61      0.92      0.73        12
           1       0.60      0.15      0.24        20
           2       0.35      0.75      0.48         8

    accuracy                           0.50        40
   macro avg       0.52      0.61      0.48        40
weighted avg       0.55      0.50      0.44        40



In [34]:
param_grid = {
    "max_depth": [None, 3, 5, 7, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20, 50]
}
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1)

grid.fit(x_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

best_dt = grid.best_estimator_
pred_y=best_dt.predict(x_test)

Best Params: {'max_depth': 5, 'min_samples_split': 10}
Best CV Accuracy: 0.675


In [35]:
print("Accuracy :",accuracy_score(y_test, pred_y))

Accuracy : 0.8


In [36]:
print("Precision:", precision_score(y_test, pred_y,average='macro'))

Precision: 0.7929292929292929


In [37]:
print("Recall   :", recall_score(y_test, pred_y,average='macro'))

Recall   : 0.8194444444444445


In [38]:
print("F1-score :", f1_score(y_test, pred_y,average='macro'))

F1-score : 0.7986270022883296


In [39]:
print("confusion matrix :\n", confusion_matrix(y_test, pred_y))

confusion matrix :
 [[10  2  0]
 [ 1 15  4]
 [ 0  1  7]]


In [41]:
comparison = pd.DataFrame({
    "Model": ["Naive Bayes", "KNN", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, pred_),
        accuracy_score(y_test, pred),
        accuracy_score(y_test, pred_y)
    ],
    "Precision": [
        precision_score(y_test, pred_, zero_division=0,average='macro'),
        precision_score(y_test, pred, zero_division=0,average='macro'),
        precision_score(y_test, pred_y, zero_division=0,average='macro')
    ],
    "Recall": [
        recall_score(y_test, pred_, zero_division=0,average='macro'),
        recall_score(y_test, pred, zero_division=0,average='macro'),
        recall_score(y_test, pred_y, zero_division=0,average='macro')
    ],
    "F1": [
        f1_score(y_test, pred_, zero_division=0,average='macro'),
        f1_score(y_test, pred, zero_division=0,average='macro'),
        f1_score(y_test, pred_y, zero_division=0,average='macro')
    ]
}).sort_values("F1", ascending=False)
print("----------------MODEL COMPARISON--------------------")
comparison


----------------MODEL COMPARISON--------------------


,Model,Accuracy,Precision,Recall,F1
2,Decision Tree,0.80,0.792929,0.819444,0.798627
1,KNN,0.50,0.521351,0.605556,0.484444
0,Naive Bayes,0.35,0.292581,0.486111,0.329025


# Strengths & Weaknesses

1.Naive Bayes

Strength-> Fast, works well with simple numeric features

Weaknesses-> Assumes features are independent (often not true),can underperform if relationships are complex.

2.KNN

Strength-> Good when patterns are local and data is well-scaled

Weaknesses-> Slow on large datasets, sensitive to scaling and noisy data, less interpretable

3.Decision Tree

Strength-> Easy to explain, handles non-linear relationships, no need scaling

Weaknesses-> Can overfit 

# Final Recommendation

Decision Tree have hightest accuracy:
0.	0.	0.	0.
acccuracy- 0.80

Precision- 0.792929

Recall- 0.819444

F1- 0.798627

Decision Tree is more interpretable because  explains decision logic clearly.

Recommended Model: Decision Tree Classifier